# GenAI Pipeline — Testing Notebook

LLM-based screening of publications for research category classification.
Uses Claude with prompt caching via the Anthropic Python SDK.

### 1. Imports and Configuration

In [4]:
import duckdb
import pandas as pd
import json
import random
import time
import anthropic
import os
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, Field, field_validator
from typing import Literal
from pydantic import create_model

load_dotenv()

DB_PATH = "../../patents_training.db"
OUTPUT_DIR = Path(".")
RAW_SUBS_DIR = Path("raw_subsets")

### 2. Data Inspection

In [7]:
con = duckdb.connect(DB_PATH, read_only=True)
print("Tables:")
print(con.sql("SHOW TABLES").df())

df_raw = con.sql("SELECT * FROM patents_raw").df()   # ← UPDATE: table name
con.close()

Tables:
                 name
0  patents_embeddings
1         patents_raw


In [8]:
df = df_raw[
    (df_raw["scope"] == "in") &
    (df_raw["research_category"].notna()) &
    (df_raw["research_category"] != "")
].reset_index(drop=True)

print(f"Raw shape: {df_raw.shape}")
print(f"Filtered shape (scope=in, research_category not empty): {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nResearch category distribution:")
print(df["research_category"].value_counts())
df.head()

Raw shape: (2450, 15)
Filtered shape (scope=in, research_category not empty): (1275, 15)

Columns: ['id', 'family_id', 'application_number', 'title', 'abstract', 'cpc', 'publication_year', 'jurisdiction', 'scope', 'pillar', 'subpillar', 'research_category', 'endproduct', 'ingredient', 'truncated']

Research category distribution:
research_category
End product formulation      445
Ingredient optimisation      437
Texturization methods        114
Bioprocess design            114
Strain development            45
Cell line development         35
Scaffolding                   34
Target molecule selection     28
Cell culture media            21
Crop development               2
Name: count, dtype: int64


,id,family_id,application_number,title,abstract,cpc,publication_year,jurisdiction,scope,pillar,subpillar,research_category,endproduct,ingredient,truncated
0,US-20170298457-A1,42829610,US15360298,LACTIC BACTERIUM WITH MODIFIED GALACTOKINASE E...,The present invention relates to a bacterial c...,"['A23C19/0323', 'C12N9/1205', 'A23C2220/206', ...",2017,US,in,PB,NaN,Strain development,Yoghurt and fermented dairy,NaN,False
1,EP-2473058-A1,42829610,EP10751643A,LACTIC BACTERIUM WITH MODIFIED GALACTOKINASE E...,The present invention relates to a bacterial c...,"['A23C19/0323', 'C12R2001/46', 'C12Y207/01006'...",2012,EP,in,PB,NaN,Strain development,Yoghurt and fermented dairy,NaN,False
2,EP-4424172-A3,52462090,EP24182765.8,A PROTEINACEOUS MEAT ANALOGUE HAVING AN IMPROV...,The invention concerns an extended shelf-life ...,"['A23V2002/00', 'A23V2200/20', 'A23J3/22', 'A2...",2025,EP,in,PB,NaN,End product formulation,Meat,NaN,False
3,US-11666069-B2,52462090,US17530304,Proteinaceous meat analogue having an improved...,An extended shelf-life proteinaceous meat anal...,"['A23J3/22', 'A23J3/18', 'A23V2200/20', 'A23V2...",2023,US,in,PB,NaN,End product formulation,Meat,NaN,False
4,EP-4627930-A3,52596486,EP25169128.3,VARIANTS OF CHYMOSIN WITH IMPROVED MILK-CLOTTI...,Variants of chymosin with improved milk-clotti...,"['A23C19/04', 'A23C19/041', 'C12N9/6483', 'A23...",2026,EP,in,PB,NaN,End product formulation,Cheese,NaN,False


In [9]:
df_pb = df[df["pillar"] == "PB"].reset_index(drop=True)
df_f  = df[df["pillar"] == "F"].reset_index(drop=True)
df_cm = df[df["pillar"] == "CM"].reset_index(drop=True)
df_cc = df[df["pillar"] == "CC"].reset_index(drop=True)

all_categories = sorted(df["research_category"].dropna().unique())
breakdown = pd.DataFrame(
    {pillar: grp["research_category"].value_counts().reindex(all_categories, fill_value=0)
     for pillar, grp in [("PB", df_pb), ("F", df_f), ("CM", df_cm), ("CC", df_cc)]},
    index=all_categories,
)
breakdown.index.name = "research_category"
breakdown["Total"] = breakdown.sum(axis=1)
breakdown

,PB,F,CM,CC,Total
research_category,,,,,
Bioprocess design,0,40,66,8,114
Cell culture media,0,0,21,0,21
Cell line development,0,0,35,0,35
Crop development,2,0,0,0,2
End product formulation,359,21,14,51,445
Ingredient optimisation,343,65,0,29,437
Scaffolding,0,0,34,0,34
Strain development,29,16,0,0,45
Target molecule selection,0,28,0,0,28


### 3. Balanced Subset Creation

Just taking as many as 10 from each research category within each pillar.


In [213]:
RANDOM_STATE=4

def create_balanced_sample(df, category_counts, random_state=RANDOM_STATE):
    """
    category_counts: dict mapping research_category -> n, e.g.
        {"Ingredient optimisation": 5, "End product formulation": 3, "Other": 2}
    Categories with no matching rows are skipped with a warning.
    If n exceeds available rows, all available rows are taken (with a warning).
    """
    samples = []
    for cat, n in category_counts.items():
        subset = df[df["research_category"] == cat]
        available = len(subset)
        if available == 0:
            print(f"  Warning: '{cat}' — no rows found, skipping.")
            continue
        if n > available:
            print(f"  Warning: '{cat}' — requested {n} but only {available} available, taking all.")
            n = available
        samples.append(subset.sample(n=n, random_state=random_state))
    combined = pd.concat(samples, ignore_index=True)
    return combined.sample(frac=1, random_state=random_state).reset_index(drop=True)

In [215]:
test_data_PB = create_balanced_sample(df_pb, {
    "Bioprocess design": 10,
    "Cell culture media": 10,
    "Cell line development": 10,
    "Consumer & market research": 10,
    "Crop development": 10,
    "End product formulation": 10,
    "Feedstocks": 10,
    "Food safety & quality": 10,
    "Health & nutrition": 10,
    "Impact assessments": 10,
    "Ingredient optimisation": 10,
    "Other": 10,
    "Scaffolding": 10,
    "Strain development": 10,
    "Target molecule selection": 10,
    "Texturization methods": 10,
}, random_state=RANDOM_STATE)
print(f"test_data_PB: {test_data_PB.shape}")
# test_data_PB[["id", "title", "scope", "pillar"]]

test_data_PB: (40, 15)


In [214]:
test_data_F = create_balanced_sample(df_f, {
    "Bioprocess design": 10,
    "Cell culture media": 10,
    "Cell line development": 10,
    "Consumer & market research": 10,
    "Crop development": 10,
    "End product formulation": 10,
    "Feedstocks": 10,
    "Food safety & quality": 10,
    "Health & nutrition": 10,
    "Impact assessments": 10,
    "Ingredient optimisation": 10,
    "Other": 10,
    "Scaffolding": 10,
    "Strain development": 10,
    "Target molecule selection": 10,
    "Texturization methods": 10,
}, random_state=RANDOM_STATE)
print(f"test_data_F: {test_data_F.shape}")
# test_data_F[["id", "title", "scope", "pillar"]]

test_data_F: (51, 15)


In [216]:
test_data_CM = create_balanced_sample(df_cm, {
    "Bioprocess design": 10,
    "Cell culture media": 10,
    "Cell line development": 10,
    "Consumer & market research": 10,
    "Crop development": 10,
    "End product formulation": 10,
    "Feedstocks": 10,
    "Food safety & quality": 10,
    "Health & nutrition": 10,
    "Impact assessments": 10,
    "Ingredient optimisation": 10,
    "Other": 10,
    "Scaffolding": 10,
    "Strain development": 10,
    "Target molecule selection": 10,
    "Texturization methods": 10,
}, random_state=RANDOM_STATE)
print(f"test_data_CM: {test_data_CM.shape}")
# test_data_CM[["id", "title", "scope", "pillar"]]

test_data_CM: (42, 15)


In [217]:
test_data_CC = create_balanced_sample(df_cc, {
    "Bioprocess design": 10,
    "Cell culture media": 10,
    "Cell line development": 10,
    "Consumer & market research": 10,
    "Crop development": 10,
    "End product formulation": 10,
    "Feedstocks": 10,
    "Food safety & quality": 10,
    "Health & nutrition": 10,
    "Impact assessments": 10,
    "Ingredient optimisation": 10,
    "Other": 10,
    "Scaffolding": 10,
    "Strain development": 10,
    "Target molecule selection": 10,
    "Texturization methods": 10,
}, random_state=3)
print(f"test_data_CC: {test_data_CC.shape}")
# test_data_CC[["id", "title", "scope", "pillar"]]

test_data_CC: (38, 15)


### 4. Save Subsets to Excel
Allows manual check of files selected. Consider whether those in the test sets are borderline cases or clear cut.

In [218]:
################################################################################################
# PLEASE CHANGE FILENAME TO THE RANDOM SEED USED IN create_balanced_sample() FOR REPRODUCIBILITY
################################################################################################
def save_subset(df, filename, output_dir=RAW_SUBS_DIR):
    path = output_dir / filename
    df.to_csv(path, index=False)
    print(f"Saved {len(df)} records to {path}")

#save_subset(initial_test_data, "initial_test_data_rand3.xlsx")
save_subset(test_data_PB, f"rescat_test_data_PB_rand{RANDOM_STATE}.csv")
save_subset(test_data_CM, f"rescat_test_data_CM_rand{RANDOM_STATE}.csv")
save_subset(test_data_CC, f"rescat_test_data_CC_rand{RANDOM_STATE}.csv")
save_subset(test_data_F, f"rescat_test_data_F_rand{RANDOM_STATE}.csv")

Saved 40 records to raw_subsets\rescat_test_data_PB_rand4.csv
Saved 42 records to raw_subsets\rescat_test_data_CM_rand4.csv
Saved 38 records to raw_subsets\rescat_test_data_CC_rand4.csv
Saved 51 records to raw_subsets\rescat_test_data_F_rand4.csv


### 5. Load Prompt and Select Dataset

In [220]:
AP_PILLAR = "F"  # ← CHANGE THIS: "PB", "F", "CM", "CC"

# Read subset data from file
test_data_PB = pd.read_csv(f"raw_subsets/rescat_test_data_PB_rand{RANDOM_STATE}.csv")
test_data_F = pd.read_csv(f"raw_subsets/rescat_test_data_F_rand{RANDOM_STATE}.csv")
test_data_CM = pd.read_csv(f"raw_subsets/rescat_test_data_CM_rand{RANDOM_STATE}.csv")
test_data_CC = pd.read_csv(f"raw_subsets/rescat_test_data_CC_rand{RANDOM_STATE}.csv")

DATASETS = {
    "PB": test_data_PB,
    "F":  test_data_F,
    "CM": test_data_CM,
    "CC": test_data_CC,
}
DATASET = DATASETS[AP_PILLAR]
print(f"Pillar: {AP_PILLAR} | Dataset: {DATASET.shape[0]} records")

# ONLY USED AS REQUIRED FOR RE-RUN SPECIFIC PUBLICATIONS
#ids_to_test = ['pub.1190984789', 'pub.1196086614']
#DATASET = DATASET[DATASET["id"].isin(ids_to_test)]
#DATASET = incorrect_rescat_data

Pillar: F | Dataset: 51 records


In [221]:
PROMPT_VERSION = "v4"  # ← CHANGE THIS to switch prompt version
PROMPT_PATH = f"{AP_PILLAR}/{PROMPT_VERSION}/prompt_rescat_patents_{AP_PILLAR}_{PROMPT_VERSION}.md"

In [222]:
def load_prompt(path=PROMPT_PATH):
    with open(path, "r", encoding="utf-8") as f:
        prompt_text = f.read()
    return prompt_text.strip()

system_prompt = load_prompt()
print(system_prompt)


You are an expert in alternative proteins and food technology.

Your task is to classify a patent on fermentation-based alternative proteins into a research category based on its title and abstract.

Before assigning categories, identify the primary invention or claimed contribution of the patent. The 9 specific categories below (everything except Other) are reserved for patents where that topic is the primary claimed or described contribution. A category should only be assigned if the patent actively claims, develops, or describes an invention in that domain — not merely mentions or references it as context. For example:
- A patent claiming or disclosing a novel microbial strain or culture for biomass fermentation → Strain development (the strain is the invention, even if it produces a specific ingredient)
- A patent claiming expression of a recombinant food protein in a microbial host, or characterising a novel enzyme or functional protein for its specific molecular properties → Targ

### 6. API Call with Prompt Caching

In [223]:
# API config

# Anthropic model options — pricing as of 2026-06-10.
# Verify at https://www.anthropic.com/pricing if costs may have changed.
# Model                  Input $/1M   Output $/1M   Context
# claude-haiku-4-5         $1.00         $5.00       200K
# claude-sonnet-4-6        $3.00        $15.00       1M
# claude-opus-4-8          $5.00        $25.00       1M
MODELS = {
    "haiku":  "claude-haiku-4-5",
    "sonnet": "claude-sonnet-4-6",
    "opus":   "claude-opus-4-8",
}
MODEL = MODELS["sonnet"]  # ← change this to switch model

MAX_TOKENS = 512         # max tokens in response; adjust based on expected reasoning length and cost tolerance
TEMPERATURE = 0.0        # 0.0 = deterministic; raise to ~0.3 to sample variance across REPETITIONS
CALL_DELAY = 1.0         # seconds between API calls
REQUEST_TIMEOUT = 120    # seconds before giving up on a single API call
MAX_RETRIES = 6          # retry attempts on rate-limit / transient errors
RETRY_BASE_SECONDS = 5.0  # exponential backoff base
RETRY_MAX_SECONDS = 90.0  # cap on backoff sleep

REPETITIONS = 1  # number of full runs; increase to measure output variance across runs

# ================================================================
# CHECKPOINT CONFIG
# Saves progress after each record; a run interrupted mid-way can
# be resumed without re-processing completed records.
# Set RESUME_INCOMPLETE = False to always start from scratch.
# ================================================================
CHECKPOINT_DIR = Path("checkpoints")
RESUME_INCOMPLETE = True

In [224]:
# ================================================================
# REASONING TOGGLE
# Keep True during testing — reasoning shows WHY the model decides
# as it does, which is essential for evaluating prompt quality.
# Set to False for production runs once the prompt is validated,
# to reduce token usage.
# ================================================================
INCLUDE_REASONING = True

PB_CATS = ["Crop development", "Strain development", "Ingredient optimisation", "End product formulation", "Texturization methods", "Food safety & quality", "Health & nutrition", "Other"]
F_CATS  = ["Feedstocks", "Target molecule selection", "Strain development", "Bioprocess design", "Ingredient optimisation", "End product formulation", "Texturization methods", "Food safety & quality", "Health & nutrition", "Other"]
CM_CATS = ["Cell line development", "Cell culture media", "Bioprocess design", "Scaffolding", "End product formulation", "Food safety & quality", "Health & nutrition", "Other"]
CC_CATS = ["Bioprocess design", "Scaffolding", "Ingredient optimisation", "End product formulation", "Texturization methods", "Food safety & quality", "Health & nutrition", "Other"]

PILLAR_CATS = {"PB": PB_CATS, "F": F_CATS, "CM": CM_CATS, "CC": CC_CATS}

def make_schema(cats, include_reasoning):
    # The LLM sometimes returns title-case values (e.g. "Health & Nutrition") while
    # our Literal expects sentence case. cats_map allows case-insensitive lookup so
    # the field_validator can normalise the value before Pydantic validates it.
    cats_map = {c.lower(): c for c in cats}
    cat_type = Literal[*cats]

    class _Base(BaseModel):
        # check_fields=False because "primary"/"secondary" are added by create_model, not defined here
        @field_validator("primary", "secondary", mode="before", check_fields=False)
        @classmethod
        def normalise_case(cls, v):
            if isinstance(v, str):
                return cats_map.get(v.lower(), v)
            return v

    fields = {"primary": (cat_type, ...), "secondary": (cat_type, ...)}
    if include_reasoning:
        fields["reasoning"] = (str, ...)
    return create_model("ClassificationSchema", __base__=_Base, **fields)

ClassificationSchema = make_schema(PILLAR_CATS[AP_PILLAR], INCLUDE_REASONING)
print(f"Schema built for {AP_PILLAR}: {list(PILLAR_CATS[AP_PILLAR])}")

client = anthropic.Anthropic(api_key=os.getenv("CLAUDE_API_KEY"))

def classify_publication(title, abstract, system_prompt):
    user_message = f"Title: {title}\n\nAbstract: {abstract}"
    response = client.messages.parse(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        timeout=REQUEST_TIMEOUT,
        system=[
            {
                "type": "text",
                "text": system_prompt,
                "cache_control": {"type": "ephemeral"}
            }
        ],
        messages=[
            {"role": "user", "content": user_message}
        ],
        output_format=ClassificationSchema,
    )
    return response.parsed_output

Schema built for F: ['Feedstocks', 'Target molecule selection', 'Strain development', 'Bioprocess design', 'Ingredient optimisation', 'End product formulation', 'Texturization methods', 'Food safety & quality', 'Health & nutrition', 'Other']


In [225]:
def is_retryable_error(exc: Exception) -> bool:
    markers = ["503", "UNAVAILABLE", "RESOURCE_EXHAUSTED", "429",
               "TIMEOUT", "TIMED OUT", "READTIMEOUT", "CONNECTTIMEOUT"]
    return any(m in str(exc).upper() for m in markers)

def retry_sleep_seconds(attempt: int) -> float:
    sleep = min(RETRY_MAX_SECONDS, RETRY_BASE_SECONDS * (2 ** attempt))
    jitter = random.uniform(0.0, min(3.0, sleep * 0.2))
    return sleep + jitter


### 7. Error Handling with Retry

In [226]:
def classify_with_error_handling(row, system_prompt):
    pub_id = row["id"]
    last_error = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            result = classify_publication(row["title"], row["abstract"], system_prompt)
            if result is None:
                print(f"  Parse failed for {pub_id}: model returned no structured output")
                return {"id": pub_id, "status": "parse_error", "error": "no structured output"}
            output = {f"{k}_LLM": v for k, v in result.model_dump().items()} # rename columns / keys to indicate LLM output
            output["id"] = pub_id
            output["status"] = "ok"
            return output
        except anthropic.APIError as e:
            last_error = e
            if attempt >= MAX_RETRIES:
                break
            if is_retryable_error(e):
                sleep_s = retry_sleep_seconds(attempt)
                print(f"  Retryable error (attempt {attempt + 1}/{MAX_RETRIES}): {e}. Sleeping {sleep_s:.1f}s.")
                time.sleep(sleep_s)
            else:
                break
    print(f"  API error for {pub_id}: {last_error}")
    return {"id": pub_id, "status": "api_error", "error": str(last_error)}


### 8. Checkpoint Helpers

In [227]:
def get_checkpoint_path(run_idx: int) -> Path:
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    return CHECKPOINT_DIR / f"run_{run_idx}_checkpoint.json"

def save_checkpoint(run_idx: int, completed_results: list) -> None:
    path = get_checkpoint_path(run_idx)
    payload = {
        "run_idx": run_idx,
        "completed_ids": [r["id"] for r in completed_results],
        "results": completed_results,
    }
    path.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")

def load_checkpoint(run_idx: int):
    path = get_checkpoint_path(run_idx)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None

def delete_checkpoint(run_idx: int) -> None:
    path = get_checkpoint_path(run_idx)
    if path.exists():
        path.unlink()


### 9. Run on Test Data

In [228]:
all_results = []

for rep in range(REPETITIONS):
    run_idx = rep + 1
    print(f"\n{'='*50}\nRun {run_idx} / {REPETITIONS}\n{'='*50}")

    checkpoint = load_checkpoint(run_idx) if RESUME_INCOMPLETE else None
    if checkpoint:
        completed_results = checkpoint["results"]
        completed_ids = set(checkpoint["completed_ids"])
        print(f"  Resuming: {len(completed_ids)} records already processed.")
    else:
        completed_results, completed_ids = [], set()

    remaining = DATASET[~DATASET["id"].isin(completed_ids)]
    total = len(DATASET)

    for _, row in remaining.iterrows():
        n_done = len(completed_results)
        print(f"  [{n_done + 1}/{total}] {row['id']}")
        result = classify_with_error_handling(row, system_prompt)
        result["run"] = run_idx
        completed_results.append(result)
        save_checkpoint(run_idx, completed_results)
        if n_done + 1 < total:
            time.sleep(CALL_DELAY)

    delete_checkpoint(run_idx)
    all_results.extend(completed_results)

results_df = pd.DataFrame(all_results)
print(f"\nCompleted: {len(results_df)} records across {REPETITIONS} run(s)")
print(f"Successful: {(results_df['status'] == 'ok').sum()}")
print(f"Errors: {(results_df['status'] != 'ok').sum()}")
results_df



Run 1 / 1
  [1/51] AR-130716-A1
  [2/51] BE-1031980-B1
  [3/51] FR-3158214-A1
  [4/51] GB-2603227-A
  [5/51] EP-3897693-A1
  [6/51] ES-2975155-A1
  [7/51] WO-2025073700-A1
  [8/51] EP-4533954-A1
  [9/51] GB-2641787-A
  [10/51] EP-4451897-A1
  [11/51] KR-20250012647-A
  [12/51] CL-2026000434-A1
  [13/51] JP-2025533662-A
  [14/51] WO-2025252839-A1
  [15/51] US-20250134121-A1
  [16/51] EP-4754234-A1
  [17/51] US-12495812-B2
  [18/51] US-20250241331-A1
  [19/51] SE-2430062-A1
  [20/51] JP-2026513110-A
  [21/51] EP-4368027-A1
  [22/51] US-11673940-B2
  [23/51] US-20230080653-A1
  [24/51] FR-3158212-A1
  [25/51] WO-2025229303-A1
  [26/51] US-12600940-B2
  [27/51] US-20250027033-A1
  [28/51] EP-4507513-A1
  [29/51] US-20250340825-A1
  [30/51] GB-2595643-A
  [31/51] EP-4298912-A1
  [32/51] JP-2026501609-A
  [33/51] EP-4476349-A2
  [34/51] US-20240247227-A1
  [35/51] US-20250206792-A1
  [36/51] EP-4642904-A1
  [37/51] US-12588687-B2
  [38/51] GB-2634922-A
  [39/51] EP-4276170-A1
  [40/51] WO-2

,primary_LLM,secondary_LLM,reasoning_LLM,id,status,run
0,Target molecule selection,Ingredient optimisation,The patent's primary novelty is the recombinan...,AR-130716-A1,ok,1
1,Bioprocess design,End product formulation,The patent's primary contribution is a novel m...,BE-1031980-B1,ok,1
2,End product formulation,Bioprocess design,The patent centers on a new food product based...,FR-3158214-A1,ok,1
3,Bioprocess design,Feedstocks,The patent's primary contribution is a system ...,GB-2603227-A,ok,1
4,Target molecule selection,Health & nutrition,The patent's primary contribution is the ident...,EP-3897693-A1,ok,1
5,End product formulation,Health & nutrition,The patent claims a finished meat analogue pro...,ES-2975155-A1,ok,1
6,Texturization methods,Target molecule selection,The patent's primary contribution is a cold-ge...,WO-2025073700-A1,ok,1
7,Target molecule selection,Bioprocess design,The patent centers on producing casein (a spec...,EP-4533954-A1,ok,1
8,End product formulation,Bioprocess design,The patent primarily claims a finished vegan y...,GB-2641787-A,ok,1
9,End product formulation,Texturization methods,The patent claims a specific composition of te...,EP-4451897-A1,ok,1


In [229]:
result_cols = ["id", "run", "primary_LLM", "secondary_LLM", "status"]
if INCLUDE_REASONING:
    result_cols.append("reasoning_LLM")

comparison = DATASET[["id", "title", "abstract", "pillar", "research_category"]].merge(
    results_df[result_cols], on="id", how="left"
)

comparison["primary_correct"] = comparison["research_category"] == comparison["primary_LLM"]
comparison["either_correct"]  = (
    (comparison["research_category"] == comparison["primary_LLM"]) |
    (comparison["research_category"] == comparison["secondary_LLM"])
)

# Overall metrics
n = len(comparison)
print(f"Top-1 accuracy (primary match):  {comparison['primary_correct'].mean():.0%}  (n={n})")
print(f"Top-2 accuracy (either match):   {comparison['either_correct'].mean():.0%}  (n={n})")

# Per-category breakdown
cat_stats = (
    comparison.groupby("research_category")
    .agg(
        n=("primary_correct", "count"),
        primary_correct=("primary_correct", "sum"),
        top2_correct=("either_correct", "sum"),
    )
    .assign(
        primary_acc=lambda d: (d["primary_correct"] / d["n"]).map("{:.0%}".format),
        top2_acc=lambda d: (d["top2_correct"] / d["n"]).map("{:.0%}".format),
    )
)
display(cat_stats)

# Detail table
display_cols = ["id", "title", "abstract", "pillar", "research_category",
                "primary_LLM", "secondary_LLM"]
if INCLUDE_REASONING:
    display_cols.append("reasoning_LLM")
display_cols += ["primary_correct", "either_correct"]
comparison[display_cols]

Top-1 accuracy (primary match):  76%  (n=51)
Top-2 accuracy (either match):   84%  (n=51)


,n,primary_correct,top2_correct,primary_acc,top2_acc
research_category,,,,,
Bioprocess design,10,8,10,80%,100%
End product formulation,10,10,10,100%,100%
Ingredient optimisation,10,4,5,40%,50%
Strain development,10,9,9,90%,90%
Target molecule selection,10,7,8,70%,80%
Texturization methods,1,1,1,100%,100%


,id,title,abstract,pillar,research_category,primary_LLM,secondary_LLM,reasoning_LLM,primary_correct,either_correct
0,AR-130716-A1,COMPOSITIONS AND APPLICATIONS DERIVED FROM OGA...,The disclosure provided herein relates to the ...,F,Ingredient optimisation,Target molecule selection,Ingredient optimisation,The patent's primary novelty is the recombinan...,False,True
1,BE-1031980-B1,FERMENTED FOOD PRODUCT AND PRODUCTION METHODS,"The invention relates to fermented foods, more...",F,Bioprocess design,Bioprocess design,End product formulation,The patent's primary contribution is a novel m...,True,True
2,FR-3158214-A1,FOOD PRODUCT BASED ON MICROALGAE,The invention relates to the production of a n...,F,Ingredient optimisation,End product formulation,Bioprocess design,The patent centers on a new food product based...,False,False
3,GB-2603227-A,System and method for cultivating and harvesti...,Disclosed is a system 100 for cultivating and ...,F,Bioprocess design,Bioprocess design,Feedstocks,The patent's primary contribution is a system ...,True,True
4,EP-3897693-A1,SYNTHETIC AND RECOMBINANT COLLAGEN PEPTIDES HA...,The invention relates to a synthetic or recomb...,F,Target molecule selection,Target molecule selection,Health & nutrition,The patent's primary contribution is the ident...,True,True
5,ES-2975155-A1,MEAT ANALOGUE FOOD COMPRISING MYCOPROTEINS AND...,Meat analogue food that includes mycoproteins ...,F,End product formulation,End product formulation,Health & nutrition,The patent claims a finished meat analogue pro...,True,True
6,WO-2025073700-A1,COLD GELLABLE RECOMBINANT BETA-LACTOGLOBULIN A...,The present invention relates to a method for ...,F,Ingredient optimisation,Texturization methods,Target molecule selection,The patent's primary contribution is a cold-ge...,False,False
7,EP-4533954-A1,METHOD FOR PRODUCING CASEIN AND USES THEREOF,The invention pertains to the food industry an...,F,Bioprocess design,Target molecule selection,Bioprocess design,The patent centers on producing casein (a spec...,False,True
8,GB-2641787-A,Foodstuff,A vegan yoghurt comprises a filamentous fungus...,F,End product formulation,End product formulation,Bioprocess design,The patent primarily claims a finished vegan y...,True,True
9,EP-4451897-A1,COMPOSITION COMPRISING TEXTURED FUNGAL PROTEIN...,The present invention relates to a composition...,F,End product formulation,End product formulation,Texturization methods,The patent claims a specific composition of te...,True,True


### 10. Save to Excel for Prompt Debugging
To assess how well the prompt does at getting the LLM to assign scope and pillar, I need to save the comparison data, then manually review what went wrong and adjust the prompt.
None of this will make it into the final workflow.

Order of working:
1. Create a new version folder in the 1_prompt_debugging folder.
2. Copy in the previous prompt. Label it with the new version number. Make updates as required based on step 6.
3. Edit Step 10 output directory (this step) and Step 5 prompt selection and input data.
4. Run the script from steps 5-10.
5. Manually review the results. Includes both metrics and 
6. Write a text document about v1 results and what changes you want to make to the prompt. Repeat from step 1.

In [230]:
save_dir = Path(f"{AP_PILLAR}/{PROMPT_VERSION}")
save_dir.mkdir(parents=True, exist_ok=True)

summary_df = pd.DataFrame([
    {"metric": "top1_accuracy", "value": f"{comparison['primary_correct'].mean():.0%}", "n": n},
    {"metric": "top2_accuracy", "value": f"{comparison['either_correct'].mean():.0%}",  "n": n},
])

out_path = save_dir / f"{AP_PILLAR}_{PROMPT_VERSION}_{MODEL}_results.xlsx"
with pd.ExcelWriter(out_path) as writer:
    comparison[display_cols].to_excel(writer, sheet_name="results",     index=False)
    cat_stats.to_excel(             writer, sheet_name="by_category")
    summary_df.to_excel(            writer, sheet_name="summary",       index=False)

print(f"Saved to {out_path}")

Saved to F\v4\F_v4_claude-sonnet-4-6_results.xlsx


In [231]:
# Records where primary_LLM did not match research_category — for re-run with modified prompt
incorrect_ids = comparison.loc[~comparison["primary_correct"], "id"]
incorrect_rescat_data = DATASET[DATASET["id"].isin(incorrect_ids)].reset_index(drop=True)
incorrect_rescat_data

,id,family_id,application_number,title,abstract,cpc,publication_year,jurisdiction,scope,pillar,subpillar,research_category,endproduct,ingredient,truncated
0,AR-130716-A1,94475360,ARP230102694,COMPOSITIONS AND APPLICATIONS DERIVED FROM OGA...,The disclosure provided herein relates to the ...,NaN,2025,AR,in,F,PF,Ingredient optimisation,Meat,Flavours and aromas,True
1,FR-3158214-A1,94384205,FR2500414,FOOD PRODUCT BASED ON MICROALGAE,The invention relates to the production of a n...,"['A23K10/16', 'A23J3/20', 'A21D2/267', 'A23J3/...",2025,FR,in,F,BF,Ingredient optimisation,Meat,NaN,False
2,WO-2025073700-A1,88373939,EP2024/077614,COLD GELLABLE RECOMBINANT BETA-LACTOGLOBULIN A...,The present invention relates to a method for ...,"['A23C21/08', 'A23C21/00', 'A23C1/12', 'A23J1/...",2025,WO,in,F,PF,Ingredient optimisation,Milk and milk proteins,"Emulsions, gels, and binders",False
3,EP-4533954-A1,88600542,EP23306702.4,METHOD FOR PRODUCING CASEIN AND USES THEREOF,The invention pertains to the food industry an...,"['A23C20/00', 'A23J1/008', 'C07K14/4732', 'C12...",2025,EP,in,F,PF,Bioprocess design,Milk and milk proteins,NaN,False
4,JP-2025533662-A,88412235,JP2025519952,Compositions and additives derived from Ogatae...,[Problem] To provide a food composition that u...,"['A23L23/00', 'A23J3/04', 'A61K2236/53', 'A61K...",2025,JP,in,F,PF,Strain development,Meat,Flavours and aromas,False
5,US-12495812-B2,74673249,US17798153,"Lipases, compositions, methods and uses thereof","The present invention relates to a wild-type, ...","['A23C13/16', 'A23C9/1216', 'A23C19/0328', 'A2...",2025,US,in,F,PF,Ingredient optimisation,Cheese,NaN,False
6,SE-2430062-A1,94688059,SE2430062A,Efficient production of recombinant casein,The present invention relates to a method for ...,"['A23J1/202', 'A23J3/20', 'A23V2002/00', 'A23J...",2025,SE,in,F,PF,Target molecule selection,Milk and milk proteins,NaN,False
7,EP-4368027-A1,84369630,EP22383070.4,PROCESS FOR THE MANUFACTURE OF FOOD PRODUCTS W...,The present invention refers to a process for ...,"['C12N1/16', 'C12R2001/72', 'A23J1/001', 'C12R...",2024,EP,in,F,BF,Ingredient optimisation,Cross-cutting,NaN,False
8,NL-2037130-B1,90362923,NL2037130,MICROBIAL LIPID COMPOSITION,MICROBIAL LIPID COMPOSITION ABSTRACT The curre...,"['A23D9/007', 'C12N1/16', 'C11B1/10', 'A23L33/...",2025,NL,in,F,PF,Target molecule selection,Cross-cutting,Fats and oils,False
9,EP-4522722-A1,82742590,EP23738438.3,PRODUCTION OF A FUNGAL FERMENTATION MEDIUM FRO...,The present invention relates to a method for ...,"['C12N1/22', 'C12R2001/645', 'C12N1/14', 'A23J...",2025,EP,in,F,NaN,Bioprocess design,Cross-cutting,NaN,False
